[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# El ecosistema *LangChain* v1: paquetes, modelos y salida estructurada.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 2: *El ecosistema LangChain v1*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se recorrerá el mapa de paquetes de *LangChain* 1.x, se inicializarán modelos de distintos proveedores con la misma interfaz, se usarán los bloques de contenido estándar y se obtendrá salida estructurada con *Pydantic*.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del curso): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del curso (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue `TUTORIAL_SECRETOS.md`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este curso usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue TUTORIAL_SECRETOS.md: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from typing import Literal

from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

---
## 1. Mapa de paquetes.

| Paquete | Qué contiene | Cuándo lo importas |
|---|---|---|
| `langchain` | `create_agent`, *middleware*, `init_chat_model`, `init_embeddings`, `@tool` | Siempre: es la entrada principal |
| `langchain-core` | Interfaces base: mensajes, *runnables*, herramientas, *vector stores* | Para tipos y utilidades de bajo nivel |
| `langchain-<proveedor>` | Integraciones: `langchain-google-genai`, `langchain-openai`, … | Indirectamente, vía `"proveedor:modelo"` |
| `langgraph` | El *runtime*: grafos, estado, *checkpointers*, interrupciones | Cuando necesitas un flujo propio (`09`) |
| `langsmith` | Trazas, *datasets* y evaluaciones | Observabilidad y evaluación (`13`) |
| `deepagents` | Agentes con planificación, archivos y subagentes | Tareas largas (`12`) |
| `langchain-classic` | Código heredado (`AgentExecutor`, `LLMChain`) | Solo para migrar código viejo |

**La relación clave:** `create_agent` compila a un grafo de *LangGraph*, y *Deep Agents* se construye sobre `create_agent`. Siempre estás usando *LangGraph* por debajo.

> ⚠️ **Tutoriales viejos.** Antes de la versión 1.0 (octubre de 2025) se usaban `AgentExecutor`, `initialize_agent` y `langgraph.prebuilt.create_react_agent`. Todo eso es código heredado: si un tutorial lo usa, está desactualizado.

---
## 2. Modelos con una sola interfaz.

> 📌 **Modelo del curso.** En todos los *notebooks* se usa *Gemini* (`MODELO = "google_genai:gemini-3.5-flash"`). Esta sección muestra que cambiar de proveedor es cambiar esa cadena; si no tienes claves de *OpenAI* o *Anthropic*, las celdas de comparación simplemente las omiten. Para usar otro modelo en todo el curso, define la variable de entorno `MODELO` (y `EMBEDDINGS`) con el formato `"proveedor:modelo"`, por ejemplo como secreto de Codespaces.

* La siguiente celda inicializará el modelo con la cadena `"proveedor:modelo"` y le hará una pregunta.

In [ ]:
modelo = init_chat_model(MODELO)
respuesta = modelo.invoke("Explica en una oración qué es un agente de IA.")
respuesta.text

* La siguiente celda mostrará los **bloques de contenido estándar** y el uso de tokens. El formato es el mismo con cualquier proveedor.

In [ ]:
respuesta.content_blocks, respuesta.usage_metadata

**Observa:** `content_blocks` normaliza el contenido (texto, razonamiento, imágenes, *tool calls*) a un formato común. Así el código que procesa respuestas no depende del proveedor.

* La siguiente celda listará los modelos para los que tienes clave configurada.

In [ ]:
import os

CANDIDATOS = {
    "google_genai:gemini-3.5-flash": "GOOGLE_API_KEY",
    "openai:gpt-5.5": "OPENAI_API_KEY",
    "anthropic:claude-sonnet-5-5": "ANTHROPIC_API_KEY",
}
disponibles = [m for m, clave in CANDIDATOS.items() if os.environ.get(clave)]
disponibles

* La siguiente celda hará la misma pregunta a cada modelo disponible.

In [ ]:
for nombre in disponibles:
    r = init_chat_model(nombre).invoke("¿Cuál es la capital de Jalisco? Responde en una palabra.")
    print(f"{nombre:35} {r.text!r:20} tokens={r.usage_metadata.get('total_tokens')}")

**💡 Ideas clave de la sección 2:**
* Cambiar de proveedor es cambiar una cadena; el resto del código no cambia.
* `content_blocks` y `usage_metadata` tienen el mismo formato en todos los proveedores.

---
## 3. Salida estructurada.

Cuando el resultado lo va a consumir **código** (no una persona), se pide un objeto validado en lugar de texto libre. Con un modelo directo se usa `with_structured_output`; con un agente, el parámetro `response_format` de `create_agent` (`04`).

Hay dos estrategias: la del **proveedor** (el modelo garantiza el esquema de forma nativa) y la de **herramienta** (el esquema se presenta como una herramienta que el modelo debe llamar). *LangChain* elige automáticamente la del proveedor cuando está disponible.

* La siguiente celda definirá un esquema con *Pydantic* y clasificará una solicitud.

In [ ]:
class Clasificacion(BaseModel):
    tipo: Literal["duda_politica", "incidente", "otro"]
    prioridad: Literal["baja", "normal", "urgente"]
    resumen: str = Field(description="Resumen de la solicitud en una oración")


clasificador = init_chat_model(MODELO).with_structured_output(Clasificacion)
clasificador.invoke("Me robaron la laptop en el aeropuerto y tenía datos de clientes.")

### Ejercicio 1 · Esquema para tickets.

Define un esquema `TicketPropuesto` con los campos `titulo` (máximo 60 caracteres), `categoria` (`hardware`, `software`, `seguridad`, `rrhh` u `otro`), `prioridad` y `requiere_humano: bool`. Úsalo para procesar las tres solicitudes de la lista `SOLICITUDES`.

**🎯 Objetivo:** diseñar un esquema que otro sistema pueda consumir. **Qué se espera aprender:** que `Field(description=...)` y los `Literal` guían al modelo tanto como el *prompt*.

* La siguiente celda muestra una solución posible.

In [ ]:
SOLICITUDES = [
    "No puedo entrar a la VPN desde ayer.",
    "Quiero instalar Tableau para un análisis.",
    "Mi jefe me pidió cambiar mi horario y no estoy de acuerdo.",
]


class TicketPropuesto(BaseModel):
    titulo: str = Field(max_length=60, description="Título breve del ticket")
    categoria: Literal["hardware", "software", "seguridad", "rrhh", "otro"]
    prioridad: Literal["baja", "normal", "urgente"]
    requiere_humano: bool = Field(description="True si la solicitud necesita criterio de una persona")


extractor = init_chat_model(MODELO).with_structured_output(TicketPropuesto)
[extractor.invoke(s) for s in SOLICITUDES]

### Ejercicio 2 · Comparar modelos.

Si tienes claves de dos proveedores, clasifica las mismas tres solicitudes con ambos. ¿Coinciden en categoría y prioridad? ¿Cuál usó menos tokens? Si solo tienes una clave, compara `gemini-3.5-flash` con otro modelo del mismo proveedor.

**🎯 Objetivo:** comparar modelos con el mismo código. **Qué se espera aprender:** que la elección de modelo es una decisión de costo y calidad que se mide, no se adivina.

**✅ Respuesta sugerida:**

Depende de los modelos disponibles. Lo esperado: coinciden en los casos claros (VPN → `software` o `hardware`; Tableau → `software`) y difieren en el ambiguo (horario → `rrhh`, con `requiere_humano=True` en el modelo más capaz). El modelo más pequeño suele usar menos tokens y ser más rápido; la diferencia de calidad se nota en el caso ambiguo.

### Reto 1 · Procesar en lote.

Usa `extractor.batch(SOLICITUDES, config={"max_concurrency": 2})` y compara el tiempo con el procesamiento uno por uno (`%%time` o `time.perf_counter`).

**🎯 Objetivo:** usar el procesamiento en lote. **Qué se espera aprender:** que `batch` paraleliza llamadas y que `max_concurrency` protege los límites de la API.

* La siguiente celda muestra una solución posible.

In [ ]:
import time

inicio = time.perf_counter()
uno_a_uno = [extractor.invoke(s) for s in SOLICITUDES]
t_secuencial = time.perf_counter() - inicio

inicio = time.perf_counter()
en_lote = extractor.batch(SOLICITUDES, config={"max_concurrency": 2})
t_lote = time.perf_counter() - inicio

round(t_secuencial, 2), round(t_lote, 2)

---
## ✅ Cierre.

**Práctica:** terminar los ejercicios 1 y 2.

**Siguiente *notebook*:** `04_primer_agente_con_create_agent.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>